In [2]:
import gc

import h5py
import os
import sys
import matplotlib.pyplot as plt

from src import loader, zdF, roi, timesync
import glob
from nre.io import load, save
import numpy as np
from tqdm import tqdm, trange
import nibabel as nib
from src import visualizations
import xarray as xr
from src.visualizations import plot_spatial_location


In [2]:
all_exps = glob.glob('/Volumes/AhmedLab/princess/data/pIP10/processed/*win01*') + glob.glob('/Volumes/AhmedLab/princess/data/pIP10/processed/*win02*')
print(all_exps)

['/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260427_GC8m_fly09_win01_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260430_GC8m_fly11_win01_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260430_GC8m_fly11_win01_trial-002', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260427_GC8m_fly09_win02_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260430_GC8m_fly11_win02_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly01_win02_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly02_win02_trial-001', '/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly03_win02_trial-001']


In [3]:
processed_experiment = all_exps[5]
all_nii = glob.glob(processed_experiment + '/motion_corrected_*.nii')
print(f'for experiment {processed_experiment}, there are {len(all_nii)} nii files')
neuron_start = 1
neuron_end = 9


for experiment /Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly01_win02_trial-001, there are 1158 nii files


In [4]:
vols_tocluster = 100
print(f'clustering based on {vols_tocluster} volumes')
# initialize empty array
array_forclusters = np.empty([512, 512, 11, vols_tocluster], dtype=np.float32)
# load in memory mapped array
for i in trange(vols_tocluster):
    single_nii = f'{processed_experiment}/motion_corrected_volume{i}.nii'
    file = nib.load(single_nii)
    data = file.get_fdata()
    array_forclusters[..., i] = data
    del data
gc.collect()
array_forclusters = array_forclusters[...,neuron_start:neuron_end,:]
# cluster into 20

n_clusters = 30
print('extracting clusters')
cluster_labels = roi.extract_ROIs(array_forclusters, n_clusters)
cluster_array = np.asarray(cluster_labels)
print(cluster_array.shape)

clusters_to_plot = [[None, None]]
plot_spatial_location(clusters_to_plot, cluster_array, [array_forclusters.shape[0],array_forclusters.shape[1],array_forclusters.shape[2]])

del array_forclusters, cluster_labels

clustering based on 100 volumes


100%|██████████| 100/100 [00:36<00:00,  2.77it/s]


extracting clusters


  0%|          | 0/8 [1:13:12<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
neuron_size = neuron_end - neuron_start
all_nii = glob.glob(processed_experiment + '/motion_corrected_*.nii')
brain_array = np.empty([512, 512, neuron_size, len(all_nii)], dtype=np.float32)
for vol in trange(len(all_nii)):
    # load up a volume
    single_nii = f'{processed_experiment}/motion_corrected_volume{vol}.nii'
    file = nib.load(single_nii)
    data = file.get_fdata()
    # remove unwanted slices
    data = data[:, :, neuron_start:neuron_end]
    brain_array[..., vol] = data

df = zdF.calculate_zscoredF(brain_array, cluster_array, n_clusters)

processed_path = processed_experiment
hf = h5py.File(f'{processed_path}/{n_clusters}_signals_260707.h5', 'w')
hf.create_dataset('labels', data=cluster_array)
hf.create_dataset('df/f', data=df)
hf.close()

del cluster_array, brain_array, df

In [ ]:
# determine how many volumes are needed to make 20 reliable clusters?
# 100 seems sufficient
# for a set number of volumes,
#
# n_volumes = [100,200]
# for n in n_volumes:

print(f'clustering based on {vols_tocluster} volumes')
brain_array = np.empty([512, 512, 11, vols_tocluster], dtype=np.float32)
# load each into memory mapped array
for i in trange(vols_tocluster):
    single_nii = f'{processed_experiment}/motion_corrected_volume{i}.nii'
    file = nib.load(single_nii)
    data = file.get_fdata()
    brain_array[..., i] = data
print(brain_array.shape)

brain_array = brain_array[...,6:-1,:]
# cluster into 20
n_clusters = 20
print('extracting clusters')
cluster_labels = roi.extract_ROIs(brain_array, n_clusters)
cluster_array = np.asarray(cluster_labels)
print(cluster_array.shape)

clusters_to_plot = [[0,0], [1,1]]
plot_spatial_location(clusters_to_plot, cluster_array, [brain_array.shape[0],brain_array.shape[1],brain_array.shape[2]])

print('calculating zscoredF')
df = zdF.calculate_zscoredF(
    brain = brain_array,
    labels_arr=cluster_labels,
    n_clusters=n_clusters
)

In [ ]:
brain_array = np.empty([512, 512, 4, len(all_nii)], dtype=np.float32)
for vol in trange(len(all_nii)):
    # load up a volume
    single_nii = f'{processed_experiment}/motion_corrected_volume{vol}.nii'
    file = nib.load(single_nii)
    data = file.get_fdata()
    # remove unwanted slices
    data = data[:,:,6:-1]
    brain_array[..., vol] = data

df = zdF.calculate_zscoredF(brain_array, cluster_array, n_clusters)


In [ ]:
processed_path = processed_experiment
hf = h5py.File(f'{processed_path}/{n_clusters}_signals_260707.h5', 'w')
hf.create_dataset('labels', data=cluster_array)
hf.create_dataset('df/f', data=df)
hf.close()

In [ ]:
for i in range(df.shape[1]):
    plt.plot(df[0,i])

quick plots for pIP10

In [ ]:
from src import visualizations
import xarray as xr
base_data_path = '/Volumes/AhmedLab/princess/data/pIP10'
experiment_id = 'pIP10_TSeries_20260427_GC8m_fly09_win02_trial-001'
processed_path = loader.get_base_path(experiment_id=experiment_id, data_stage='processed', base_dir=base_data_path)

cluster_labels, signal, timestamps = loader.load_clusters(processed_path)

In [ ]:
signal = df

In [ ]:
n

In [ ]:
signal_xr = xr.DataArray(
    data=signal,
    dims=['zposs', 'roi', 'time'],
    coords={'time': range(signal.shape[2])}
)
dimensions = [512, 512, 4]


In [ ]:
rois = [[None, None]]
visualizations.plot_spatial_location(rois, cluster_array, dimensions)


In [ ]:
# find clusters to plot

rois = [[0,0], [3,5], [0,6], [1,0], [1,6], [1,12], [2,2], [2,4], [3,10]]
visualizations.plot_spatial_location(rois, cluster_array, dimensions)
visualizations.plot_zscored_activity(rois, signal_xr)

In [ ]:
for z in range(4):
    to_plot = [[z, x]for x in range(signal.shape[1])]
    visualizations.plot_zscored_activity(to_plot, signal_xr)
    visualizations.plot_spatial_location(to_plot, cluster_array, dimensions)

In [ ]:
from src.qc import stitch_moco_volumes
import glob
all_exps =glob.glob('/Volumes/AhmedLab/princess/data/pIP10/processed/*win02*')
# print(all_exps)

for i in all_exps[-3:]:
    print(i)
    stitch_moco_volumes(i, n_volumes=1000)

/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly01_win02_trial-001


100%|██████████| 1000/1000 [07:17<00:00,  2.28it/s]


/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly02_win02_trial-001


100%|██████████| 1000/1000 [07:45<00:00,  2.15it/s]


/Volumes/AhmedLab/princess/data/pIP10/processed/pIP10_TSeries_20260602_GC8mChrR_fly03_win02_trial-001


 46%|████▌     | 455/1000 [06:49<03:25,  2.65it/s]

In [5]:
import nibabel as nib
import time
from sys import getsizeof
import numpy as np

full = nib.load('/Volumes/AhmedLab/princess/data/raw/pIP10_TSeries_20260421_GC8m_fly02_win01_trial0-001/pIP10_TSeries_20260421_GC8m_fly02_win01_trial0-001_channel_2.nii')
print( getsizeof(full))

48


In [6]:
data = full.dataobj[...,0]
print(f'dataobj shape: {data.shape}, {getsizeof(data)}')
data_arr = np.asarray(data)
print(f'data_arr shape: {data_arr.shape}, {getsizeof(data_arr)}')

data_mmap

dataobj shape: (512, 512, 11), 144
data_arr shape: (512, 512, 11), 144
